## Investigator decision: mortality winner accepted
The investigator accepted the current mortality winner and its documented encoding after reviewing the fixed-parameter comparison. The dated winner bundle and eTable 2 remain unchanged. Notebook 03 may now be prepared using this winner and the already accepted readmission winner. This decision closes the parameter-selection checkpoint; it does not assert stability of later SHAP rules, Cox specifications or holdout metrics. Full cohort SHAP still requires its resource review.


## Scope and approved decisions
The original development cohort and held-out split are fixed. This notebook tunes **mortality**, independently of readmission model selection. It reuses the five datasets imputed by notebook 00 on complete development without outcomes, as explicitly selected by the investigator. No imputer is fitted within a CV fold.
The method actually executed in March is preserved: 100 NSGA-II trials, cyclic imputation assignment, pruning after at least two folds when mean horizon C is below 0.60, and logarithmic L2 search from 0.50 to 15.00. Up to 20 Pareto candidates are evaluated across five folds and all five imputations; final ties use global C and between-imputation C dispersion.
The arithmetic mean of horizon Brier scores is retained with the investigator's stated computational rationale. It is **not a time-integrated Brier score**. The preliminary random search and unused optimism bootstrap do not feed selection and remain omitted; see the simplification log.
Notebook 01 supplies named, verified helpers; its tuning is never executed here. This notebook uses Python/surv only. The held-out sample is excluded from tuning. Encoding still learns from each training fold; the different scope of imputation is stated below. The candidate universe, quality exclusions, cohort rules and search ranges remain inherited decisions.
**Retained limitation of internal CV.** The five missRanger imputations were fitted once on the complete development sample, without outcome or follow-up variables, before survival-model cross-validation. Predictors from each internal validation fold therefore contributed to learning the imputer, although that fold is excluded from fitting its survival model. This CV does not independently validate the complete preprocessing-and-model-selection workflow. missRanger OOB diagnostics assess the imputation forests and do not remove this dependence. The direction and magnitude of its effect on the retained workflow have not been isolated by the available comparisons. The investigator explicitly retained this scope on 2026-10-05.
**Separate holdout boundary.** Holdout rows and outcomes were excluded from fitting the development imputers, ML tuning and SHAP screening. However, notebook 00's descriptive early-death audit inspected holdout outcomes, and inherited discovery decisions remain documented. Do not describe the entire historical and reconstructed workflow as wholly independent of the holdout.

In [1]:
#| label: death-config-and-environment
#| message: false
import time
t0 = time.perf_counter()
# Resolve all machine and input paths from the single project configuration.
import os, sys, json, platform, importlib.metadata, gc, hashlib, pickle, functools, datetime, html
from pathlib import Path
root = Path.cwd().resolve()
while not (root / "config.yml").is_file():
    if root.parent == root:
        raise FileNotFoundError("config.yml was not found.")
    root = root.parent
import yaml
config = yaml.safe_load((root / "config.yml").read_text(encoding="utf-8"))
settings = config["xgboost_mortality"]
threads = int(config["imputation"]["threads"])
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = str(threads)
def resolve_path(value):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (root / path).resolve()
def source_path(name):
    return resolve_path(config["sources"][name])
assert Path(sys.prefix).name == config["runtime"]["xgboost_environment"], "Select the configured surv kernel."
mode = settings["mode"]
assert mode in ("preflight", "pilot", "full"), "Invalid execution mode."
# Import the installed scientific stack after limiting native thread pools.
import numpy as np
import pandas as pd
import pyarrow as pa
pa.set_cpu_count(threads)
pa.set_io_thread_count(threads)
import psutil
import xgboost as xgb
from sksurv.linear_model.coxph import BreslowEstimator
from sksurv.metrics import brier_score, concordance_index_ipcw
from sksurv.util import Surv
from sksurv.nonparametric import CensoringDistributionEstimator
from sklearn.model_selection import StratifiedKFold
import optuna
from threadpoolctl import threadpool_limits, threadpool_info
from IPython.display import HTML, display
thread_limits = threadpool_limits(limits=threads)
seed = int(config["imputation"]["seed"])
np.random.seed(seed)
horizons = np.asarray(settings["horizons_months"], dtype=float)
assert settings["brier_aggregation"] == "arithmetic_mean"
assert settings["probability_target"] == "net_survival_breslow"
assert threads == 2 and np.all(np.diff(horizons) > 0)
written = []
run_started = time.perf_counter()
process = psutil.Process()
start_rss_gib = process.memory_info().rss / 2**30
# Keep exploratory tables scrollable without adding a widget dependency.
def show_table(data, caption):
    safe_caption = html.escape(caption)
    table = data.to_html(index=False, escape=True, float_format=lambda x: f"{x:.6f}")
    display(HTML(f'<div role="region" tabindex="0" aria-label="{safe_caption}" style="max-height:420px;overflow:auto;max-width:100%;border:1px solid #ddd;padding:0.4rem"><strong>{safe_caption}</strong>{table}</div>'))
show_table(pd.DataFrame({"Component": ["Kernel", "Seed", "Native threads", "Available RAM (GiB)", "Brier aggregation", "Probability target"], "Value": [f"Python {platform.python_version()} / {Path(sys.prefix).name}", seed, threads, round(psutil.virtual_memory().available / 2**30, 3), settings["brier_aggregation"], settings["probability_target"]]}), "Execution contract")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

C:\Users\andre\miniconda3\envs\surv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Component,Value
Kernel,Python 3.11.14 / surv
Seed,2125
Native threads,2
Available RAM (GiB),54.370000
Brier aggregation,arithmetic_mean
Probability target,net_survival_breslow


Elapsed: 0.024 minutes


## Shared helpers and probability controls
Only the indicated helper cells are loaded from notebook 01; all variables are created in this fresh kernel. Mortality retains the original probability clipping interval and is checked against its own historical Breslow implementation before cohort tuning.

In [2]:
#| label: death-load-shared-functions
#| message: false
t0 = time.perf_counter()
# Load named cells from the preceding notebook without executing its tuning stages.
shared_notebook = json.loads(source_path("readmission_notebook").read_text(encoding="utf-8"))
shared_sources = {}
for cell in shared_notebook["cells"]:
    source = "".join(cell.get("source", []))
    if cell["cell_type"] == "code" and source.startswith("#| label: "):
        label = source.splitlines()[0].split(": ", 1)[1]
        if label in shared_sources:
            raise ValueError(f"Duplicate shared cell label: {label}")
        shared_sources[label] = source
def run_shared(label):
    exec(compile(shared_sources[label], f"{source_path('readmission_notebook').name}:{label}", "exec"), globals())
shared_function_labels = ["readm-predict-net-survival", "readm-encode-zero-time-labels", "readm-synthetic-probability-controls", "readm-completed-data-and-artifacts", "readm-training-only-encoding"]
assert set(shared_function_labels).issubset(shared_sources)
run_shared("readm-predict-net-survival")
run_shared("readm-encode-zero-time-labels")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Net survival and mean Brier functions ready; no cohort model fitted.
Elapsed: 0.000 minutes
Zero-time encoding changes only the XGBoost interface, for both events and censoring.
Elapsed: 0.000 minutes
Elapsed: 0.000 minutes


In [3]:
#| label: death-verify-breslow-and-label-controls
#| message: false
t0 = time.perf_counter()
# Preserve mortality's original probability clipping after train-only Breslow estimation.
def predict_mortality_survival(y_train, margin_train, margin_valid, times):
    return np.clip(predict_net_survival(y_train, margin_train, margin_valid, times), 1e-6, 1.0)
run_shared("readm-synthetic-probability-controls")
# Extract only the historical helper definition; no original cohort code is executed.
import ast
original_notebook = json.loads(source_path("mortality_original_notebook").read_text(encoding="utf-8"))
legacy_phase1 = ["".join(cell["source"]) for cell in original_notebook["cells"] if "".join(cell["source"]).startswith("# @title Phase 1 - Death Optuna Multi-Objective")]
assert len(legacy_phase1) == 1
legacy_nodes = [node for node in ast.parse(legacy_phase1[0]).body if isinstance(node, ast.FunctionDef) and node.name == "predict_survival_probs_breslow"]
assert len(legacy_nodes) == 1
legacy_namespace = {"np": np}
exec(compile(ast.Module(body=legacy_nodes, type_ignores=[]), "historical-mortality-breslow", "exec"), legacy_namespace)
legacy_control = legacy_namespace["predict_survival_probs_breslow"](synthetic_y, synthetic_risk, np.exp(synthetic_valid_margin), synthetic_times)
new_control = predict_mortality_survival(synthetic_y, synthetic_margin, synthetic_valid_margin, synthetic_times)
assert np.allclose(legacy_control, new_control, atol=1e-14, rtol=1e-12)
show_table(pd.DataFrame([{"Control": "Mortality Breslow versus original helper, including clipping", "Maximum absolute difference": float(np.max(np.abs(legacy_control - new_control))), "Reportable study result": False}]), "Historical probability anchor")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Control,Result
Breslow versus hand calculation: maximum absolute error,1.11e-16
Brier versus direct MSE: maximum absolute error,0
Common margin shift,passed
"Bounded, nonincreasing survival",passed
Invalid probabilities and unsupported horizon,rejected


Elapsed: 0.000 minutes


Control,Maximum absolute difference,Reportable study result
"Mortality Breslow versus original helper, including clipping",0.000000,False


Elapsed: 0.000 minutes


## Historical selection and search contract
The reference is the final export linked to the March 5 study ending at 20:13. The earlier same-day study is a separate run; trial counts are not added. All historical results below are references, not results of the clean pipeline.

In [4]:
#| label: death-verify-historical-selection
#| message: false
t0 = time.perf_counter()
# Reproduce the final historical ranking, including mortality's two tie-breaks.
reference = pd.read_csv(source_path("xgboost_mortality_reference"))
c_column, b_column = "Phase2_Multi_Horizon_C_Index", "Phase2_IBS"
reference["recomputed_distance"] = distance_to_ideal(reference[c_column], reference[b_column])
assert np.allclose(reference["recomputed_distance"], reference["Distance_to_Ideal"], atol=1e-14, rtol=0)
reference = reference.sort_values(["recomputed_distance", "Phase2_Global_C_Index", "C_Index_SD_across_imputations"], ascending=[True, False, True], kind="stable").reset_index(drop=True)
historical_winner = reference.iloc[0]
assert int(historical_winner["trial_id"]) == 55
search_ranges = {"learning_rate": ("float", 0.003, 0.05, True), "max_depth": ("int", 3, 7, False), "min_child_weight": ("int", 3, 40, False), "subsample": ("float", 0.60, 0.85, False), "colsample_bytree": ("float", 0.40, 0.65, False), "reg_alpha": ("float", 0.0001, 5.0, True), "reg_lambda": ("float", 0.5, 15.0, True), "gamma": ("float", 0.0, 1.0, False)}
param_names = list(search_ranges)
def native_params(values):
    return {name: int(values[name]) if name in ("max_depth", "min_child_weight") else float(values[name]) for name in param_names}
# Verify executed Optuna distributions and balanced imputation assignment from the saved study.
import joblib
historical_study = joblib.load(source_path("xgboost_mortality_study"))
assert len(historical_study.trials) == 100
for trial in historical_study.trials:
    assert trial.user_attrs["Sampled_Imputation"] == trial.number % 5 + 1
    for name, (_, lower, upper, logarithmic) in search_ranges.items():
        distribution = trial.distributions[name]
        assert (distribution.low, distribution.high, distribution.log) == (lower, upper, logarithmic)
assert all(np.isclose(historical_study.trials[55].params[name], historical_winner[name], atol=0, rtol=1e-12) for name in param_names)
reference["C_squared_distance_share_percent"] = 100 * (1 - reference[c_column])**2 / reference["recomputed_distance"]**2
show_table(reference[["trial_id", c_column, b_column, "Phase2_Global_C_Index", "C_Index_SD_across_imputations", "Distance_to_Ideal", "C_squared_distance_share_percent"]], "Verified historical candidates; IBS column is an arithmetic mean")
show_table(pd.DataFrame([{"Parameter": name, "Lower": lower, "Upper": upper, "Scale": "log" if logarithmic else "linear", "Historical selected": native_params(historical_winner)[name]} for name, (_, lower, upper, logarithmic) in search_ranges.items()]), "Executed mortality search ranges")
print("C shares describe distance components, not prespecified objective weights.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

trial_id,Phase2_Multi_Horizon_C_Index,Phase2_IBS,Phase2_Global_C_Index,C_Index_SD_across_imputations,Distance_to_Ideal,C_squared_distance_share_percent
55,0.762197,0.017730,0.745860,0.000859,0.238463,99.447206
75,0.762089,0.017703,0.747055,0.000653,0.238568,99.449343
46,0.761746,0.017702,0.746341,0.000435,0.238911,99.450987
95,0.761729,0.017733,0.746544,0.001330,0.238930,99.449156
35,0.761695,0.017713,0.747041,0.000641,0.238962,99.450533


Parameter,Lower,Upper,Scale,Historical selected
learning_rate,0.003000,0.050000,log,0.030967
max_depth,3.000000,7.000000,linear,6.000000
min_child_weight,3.000000,40.000000,linear,15.000000
subsample,0.600000,0.850000,linear,0.778743
colsample_bytree,0.400000,0.650000,linear,0.616890
reg_alpha,0.000100,5.000000,log,0.013708
reg_lambda,0.500000,15.000000,log,2.536795
gamma,0.000000,1.000000,linear,0.982286


C shares describe distance components, not prespecified objective weights.
Elapsed: 0.002 minutes


## Development outcomes and completed predictors
Only development rows enter tuning. The approved positive epsilon is used solely for signed XGBoost labels at zero time; analytical follow-up, events and participants remain unchanged.

In [5]:
#| label: death-audit-zero-time-labels
#| message: false
t0 = time.perf_counter()
# Audit the original signed-time rule using development rows only.
endpoint_columns = ["original_row_id", "is_train", "death_event", "death_time_from_disch_m", "readmit_event", "readmit_time_from_disch_m"]
development_endpoint = pd.read_parquet(source_path("clean_completed_reference"), columns=endpoint_columns, filters=[("is_train", "=", True)])
assert development_endpoint["is_train"].all() and development_endpoint["original_row_id"].is_unique
assert development_endpoint[endpoint_columns].notna().all().all()
y_development = Surv.from_arrays(development_endpoint["death_event"].eq(1), development_endpoint["death_time_from_disch_m"])
assert np.isfinite(y_development["time"]).all() and (y_development["time"] >= 0).all()
original_xgb_labels = np.where(y_development["event"], y_development["time"], -y_development["time"]).astype(np.float32)
lost_events = y_development["event"] & ~(original_xgb_labels > 0)
zero_time = y_development["time"] == 0
assert np.array_equal(lost_events, y_development["event"] & zero_time)
label_check = pd.DataFrame({"Check": ["Development participants", "Observed deaths at time zero", "Censored observations at time zero", "Recorded events lost by signed-time encoding"], "Count": [len(y_development), int((zero_time & y_development["event"]).sum()), int((zero_time & ~y_development["event"]).sum()), int(lost_events.sum())]})
show_table(label_check, "Original label audit; analytical outcomes unchanged")
corrected_xgb_labels = xgboost_cox_labels(y_development)
assert np.array_equal(corrected_xgb_labels > 0, y_development["event"])
assert np.array_equal(np.argsort(np.abs(original_xgb_labels), kind="stable"), np.argsort(np.abs(corrected_xgb_labels), kind="stable"))
assert np.array_equal(original_xgb_labels[~zero_time], corrected_xgb_labels[~zero_time])
print(f"Event mismatches after the approved interface correction: {np.count_nonzero((corrected_xgb_labels > 0) != y_development['event'])}")
print("Analytical times, participants, events and time-zero ties are unchanged.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Check,Count
Development participants,70521
Observed deaths at time zero,160
Censored observations at time zero,3867
Recorded events lost by signed-time encoding,160


Event mismatches after the approved interface correction: 0
Analytical times, participants, events and time-zero ties are unchanged.
Elapsed: 0.001 minutes


In [6]:
#| label: death-load-completed-development
#| message: false
t0 = time.perf_counter()
# Reuse the verified completed-development loader; no imputation is fitted here.
run_shared("readm-completed-data-and-artifacts")
assert np.array_equal(development_data["death_event"], development_endpoint["death_event"])
assert np.array_equal(development_data["death_time_from_disch_m"], development_endpoint["death_time_from_disch_m"])
print("Mortality outcomes and row identities agree with notebook 00.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Imputation,Development rows,Imputation predictors,Retained ML predictors,Missing predictors,Source
1,70521,39,36,0,outputs/objects/completed_imp1_2026-10-02_71f0ce969d33.parquet
2,70521,39,36,0,outputs/objects/completed_imp2_2026-10-02_71f0ce969d33.parquet
3,70521,39,36,0,outputs/objects/completed_imp3_2026-10-02_71f0ce969d33.parquet
4,70521,39,36,0,outputs/objects/completed_imp4_2026-10-02_71f0ce969d33.parquet
5,70521,39,36,0,outputs/objects/completed_imp5_2026-10-02_71f0ce969d33.parquet


No R subprocess or imputation fit is run here. Metadata, including center_id, never enters the predictor matrix.
Elapsed: 0.018 minutes
Mortality outcomes and row identities agree with notebook 00.
Elapsed: 0.018 minutes


## Frozen mortality folds
Mortality retains programme-by-death stratification and the original seed. Actual memberships must match the previous mortality manifest. The five completed development datasets are the same inputs used by notebook 01, while mortality and readmission retain their own outcome-specific CV folds.
The former validation-only missingness issue for sex_rec arose during fold-wise imputation. Notebook 00 already completed that variable on full development. No additional forest, replacement rule or participant exchange is needed or implemented; all actual encodings are checked during preflight.

In [7]:
#| label: death-freeze-and-check-folds
#| message: false
t0 = time.perf_counter()
# Preserve the original fold memberships; the imputations are already complete.
plan_codes = development_data["plan_type_corr"].map({"pg-pab": 0, "pg-pr": 1, "m-pr": 2, "pg-pai": 3, "m-pai": 4})
assert plan_codes.notna().all()
strata = {"dual": plan_codes.to_numpy(dtype=int) * 10 + y_development["event"].astype(int)}
fold_manifest = pd.DataFrame({"original_row_id": development_data["original_row_id"].to_numpy()})
splits = {}
for scheme, labels in strata.items():
    counts = pd.Series(labels).value_counts()
    if len(counts) < 2 or counts.min() < 5:
        raise ValueError("The fixed cohort no longer supports the original stratification.")
    splits[scheme] = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=seed).split(np.zeros(len(labels)), labels))
    membership = np.zeros(len(labels), dtype=np.int8)
    for fold, (training, validation) in enumerate(splits[scheme], start=1):
        assert not np.intersect1d(training, validation).size
        membership[validation] = fold
    assert np.all((membership >= 1) & (membership <= 5))
    fold_manifest[f"{scheme}_fold"] = membership
# Anchor the actual memberships to the previous manifest before any new model fit.
previous_folds = pd.read_parquet(source_path("mortality_fold_reference"))
pd.testing.assert_frame_equal(fold_manifest, previous_folds[fold_manifest.columns])
data_contract = {"imputation_fit_scope": "complete_development_before_cv", "completed_sha256": completed_hashes, "imputation": config["imputation"], "seed": seed, "manifest_sha256": hashlib.sha256(fold_manifest.to_csv(index=False).encode()).hexdigest(), "loader_code": notebook_code(source_path("readmission_notebook"), ["readm-completed-data-and-artifacts"])}
data_signature = fingerprint(data_contract)
manifest_path = artifact_path("xgb_death_fold_manifest", data_signature, ".parquet")
if manifest_path.exists():
    pd.testing.assert_frame_equal(pd.read_parquet(manifest_path), fold_manifest)
else:
    manifest_path.write_bytes(fold_manifest.to_parquet(index=False))
    pd.testing.assert_frame_equal(pd.read_parquet(manifest_path), fold_manifest)
    written.append(manifest_path)
# Check censoring support at every requested horizon without fitting a survival model.
support_rows = []
for scheme, scheme_splits in splits.items():
    for fold, (training, validation) in enumerate(scheme_splits, start=1):
        y_train, y_valid = y_development[training], y_development[validation]
        censoring = CensoringDistributionEstimator().fit(y_train)
        assert np.all(censoring.predict_proba(horizons) > 0)
        assert np.isfinite(censoring.predict_ipcw(y_valid)).all()
        for horizon in horizons:
            assert np.isfinite(censoring.predict_ipcw(y_valid[y_valid["time"] < horizon])).all()
        assert y_valid["time"].min() <= horizons[0] and horizons[-1] < y_valid["time"].max()
        support_rows.append({"Scheme": scheme, "Fold": fold, "Training": len(training), "Validation": len(validation), "Changed memberships": 0, "IPCW support": "passed"})
show_table(pd.DataFrame(support_rows), "Original internal validation partitions, unchanged")
print("Every CV fold uses values prepared from complete development. Internal CV shares this preprocessing.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")


Scheme,Fold,Training,Validation,Changed memberships,IPCW support
dual,1,56416,14105,0,passed
dual,2,56417,14104,0,passed
dual,3,56417,14104,0,passed
dual,4,56417,14104,0,passed
dual,5,56417,14104,0,passed


Every CV fold uses values prepared from complete development. Internal CV shares this preprocessing.
Elapsed: 0.004 minutes


In [8]:
#| label: death-training-only-encoding
#| message: false
t0 = time.perf_counter()
# Reuse the study-defined encoding and its training-versus-validation invariance controls.
run_shared("readm-training-only-encoding")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Training encoding is invariant to validation changes; unknown ordinal values are rejected.
Elapsed: 0.014 minutes
Elapsed: 0.014 minutes


## Completed inputs and cached model evaluation
Historical quality exclusions remain fixed. Each CV fold uses its rows from one of the five completed development datasets. No imputer, extra sex forest or fold-specific completed dataset is created. Model scores retain a fingerprint of their input files, imputation scope, method and environment.
Early stopping uses the validation fold, as in the original analysis. Predictions retain all returned trees. These are development tuning estimates; they share both preprocessing and model selection and are not independent final performance estimates.

In [9]:
#| label: death-fold-cache-and-evaluation
#| message: false
t0 = time.perf_counter()
# Score models using the completed-development loader shared with notebook 02.
fixed_xgb = {"objective": "survival:cox", "eval_metric": "cox-nloglik", "tree_method": "hist", "device": "cpu", "nthread": threads, "seed": seed, "verbosity": 0}
method_labels = ["death-verify-breslow-and-label-controls", "death-verify-historical-selection", "death-audit-zero-time-labels", "death-freeze-and-check-folds", "death-fold-cache-and-evaluation", "death-optuna-and-pareto"]
model_contract = {"data_signature": data_signature, "data_contract": data_contract, "retained_predictors": retained_predictors, "horizons": horizons.tolist(), "probability_target": settings["probability_target"], "brier_aggregation": settings["brier_aggregation"], "probability_clip": [1e-6, 1.0], "zero_time_label_months": settings["zero_time_label_months"], "fixed_xgb": fixed_xgb, "rounds": 1500, "early_stopping": 30, "budgets": {"trials": 100, "rescore_top_k": 20, "folds": 5, "imputations": 5}, "pruning": {"minimum_folds": 2, "mean_C_below": 0.60}, "prediction_trees": "all_returned_trees_as_in_original", "search_ranges": search_ranges, "tie_breaks": ["Distance_to_Ideal ascending", "global_C descending", "C_SD_across_imputations ascending"], "code": notebook_code(source_path("mortality_notebook"), method_labels), "shared_code": {name: shared_sources[name] for name in shared_function_labels}, "versions": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "xgboost", "optuna", "scikit-learn", "scikit-survival"]}}
model_signature = fingerprint(model_contract)
score_path = artifact_path("xgb_death_fold_scores", model_signature, ".pkl")
if score_path.exists():
    with score_path.open("rb") as stream:
        score_cache = pickle.load(stream)
    assert score_cache["contract"] == model_contract
else:
    score_cache = {"contract": model_contract, "scores": {}}
new_model_fits = 0
def evaluate_fold(params, scheme, fold, imputation):
    global new_model_fits
    if scheme != "dual":
        raise ValueError("Mortality uses the original programme-by-death folds.")
    params = native_params(params)
    key = fingerprint({"params": params, "scheme": scheme, "fold": fold, "imputation": imputation})
    if key in score_cache["scores"]:
        return score_cache["scores"][key]
    if mode == "preflight":
        raise RuntimeError("Preflight cannot fit XGBoost models.")
    x_train, x_valid, y_train, y_valid, imputation_metadata = encoded_fold(scheme, fold, imputation)
    model_started = time.perf_counter()
    d_train = xgb.DMatrix(x_train, label=xgboost_cox_labels(y_train), nthread=threads)
    d_valid = xgb.DMatrix(x_valid, label=xgboost_cox_labels(y_valid), nthread=threads)
    booster = xgb.train(dict(fixed_xgb, **params), d_train, num_boost_round=1500, evals=[(d_valid, "validation")], early_stopping_rounds=30, verbose_eval=False)
    fitting_seconds = time.perf_counter() - model_started
    # Preserve the original all-returned-tree prediction policy after early stopping.
    rounds = booster.num_boosted_rounds()
    tree_range = (0, rounds)
    risk_valid = booster.predict(d_valid, iteration_range=tree_range)
    margin_train = booster.predict(d_train, output_margin=True, iteration_range=tree_range)
    margin_valid = booster.predict(d_valid, output_margin=True, iteration_range=tree_range)
    survival = predict_mortality_survival(y_train, margin_train, margin_valid, horizons)
    c_horizons = [float(concordance_index_ipcw(y_train, y_valid, risk_valid, tau=horizon)[0]) for horizon in horizons]
    global_c = float(concordance_index_ipcw(y_train, y_valid, risk_valid)[0])
    mean_brier, brier_horizons = mean_horizon_brier(y_train, y_valid, survival, horizons)
    if not np.isfinite(c_horizons + [global_c, mean_brier]).all():
        raise ValueError("A fold metric was nonfinite.")
    result = {"params": params, "scheme": scheme, "fold": fold, "imputation": imputation, "C": float(np.mean(c_horizons)), "mean_Brier": mean_brier, "global_C": global_c, "C_by_horizon": c_horizons, "Brier_by_horizon": brier_horizons.tolist(), "best_iteration": int(booster.best_iteration), "returned_rounds": rounds, "features": list(x_train.columns), "fit_seconds": fitting_seconds, "model_and_scoring_seconds": time.perf_counter() - model_started, "imputation_source": imputation_metadata["source"], "imputation_fit_scope": imputation_metadata["fit_scope"], "imputation_refitted": False}
    score_cache["scores"][key] = result
    save_pickle(score_cache, score_path)
    new_model_fits += 1
    del booster, d_train, d_valid, margin_train, margin_valid, risk_valid, survival
    gc.collect()
    return result
print("Mortality evaluation functions ready; no fitting is started by this cell.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Mortality evaluation functions ready; no fitting is started by this cell.
Elapsed: 0.001 minutes


In [10]:
#| label: death-optuna-and-pareto
#| message: false
t0 = time.perf_counter()
# Preserve the one-stage search, cyclic imputations, pruning and saved sampler state.
def pareto_table(study):
    rows = [{"trial_id": trial.number, "C": trial.values[0], "mean_Brier": trial.values[1], **native_params(trial.params)} for trial in study.best_trials]
    if not rows:
        raise RuntimeError("No complete Pareto candidate is available.")
    table = pd.DataFrame(rows)
    table["Distance_to_Ideal"] = distance_to_ideal(table["C"], table["mean_Brier"])
    return table.sort_values("Distance_to_Ideal", kind="stable").reset_index(drop=True)
def optimize_stage():
    if mode != "full":
        raise RuntimeError("Full mortality tuning requires execution mode full.")
    trial_budget = model_contract["budgets"]["trials"]
    study_path = artifact_path("xgb_death_optuna", model_signature, ".pkl")
    if study_path.exists():
        with study_path.open("rb") as stream:
            saved = pickle.load(stream)
        assert saved["signature"] == model_signature
        study = saved["study"]
    else:
        study = optuna.create_study(directions=["maximize", "minimize"], sampler=optuna.samplers.NSGAIISampler(seed=seed), study_name=f"mortality_{model_signature}")
    if any(trial.state in (optuna.trial.TrialState.FAIL, optuna.trial.TrialState.RUNNING) for trial in study.trials):
        raise RuntimeError("A failed or unfinished trial requires inspection before resuming.")
    def objective(trial):
        params = {}
        for name, (kind, lower, upper, logarithmic) in search_ranges.items():
            params[name] = trial.suggest_int(name, lower, upper) if kind == "int" else trial.suggest_float(name, lower, upper, log=logarithmic)
        imputation = trial.number % 5 + 1
        trial.set_user_attr("imputation", imputation)
        fold_results = []
        for fold in range(1, 6):
            fold_results.append(evaluate_fold(params, "dual", fold, imputation))
            if len(fold_results) >= 2 and np.mean([row["C"] for row in fold_results]) < 0.60:
                raise optuna.TrialPruned()
        trial.set_user_attr("global_C", float(np.mean([row["global_C"] for row in fold_results])))
        return float(np.mean([row["C"] for row in fold_results])), float(np.mean([row["mean_Brier"] for row in fold_results]))
    def checkpoint(study, trial):
        save_pickle({"signature": model_signature, "study": study}, study_path)
        print(f"Mortality trial {trial.number + 1}/{trial_budget}: {trial.state.name}", flush=True)
    finished = sum(trial.state.is_finished() for trial in study.trials)
    if finished > trial_budget:
        raise RuntimeError("Saved study exceeds the fixed trial budget.")
    if finished < trial_budget:
        study.optimize(objective, n_trials=trial_budget - finished, n_jobs=1, callbacks=[checkpoint], show_progress_bar=False)
    assert sum(trial.state.is_finished() for trial in study.trials) == trial_budget
    return study
def select_final(table):
    return table.sort_values(["Distance_to_Ideal", "global_C", "C_SD_across_imputations"], ascending=[True, False, True], kind="stable").reset_index(drop=True)
def rescore_pareto(study):
    candidates = pareto_table(study).head(model_contract["budgets"]["rescore_top_k"])
    rows, detail = [], {}
    for candidate in candidates.to_dict("records"):
        params = native_params(candidate)
        per_imputation = []
        for imputation in range(1, 6):
            records = [evaluate_fold(params, "dual", fold, imputation) for fold in range(1, 6)]
            per_imputation.append({name: float(np.mean([row[name] for row in records])) for name in ["C", "mean_Brier", "global_C"]})
        trial_id = int(candidate["trial_id"])
        detail[trial_id] = per_imputation
        rows.append({"trial_id": trial_id, **{name: float(np.mean([row[name] for row in per_imputation])) for name in ["C", "mean_Brier", "global_C"]}, "C_SD_across_imputations": float(np.std([row["C"] for row in per_imputation])), "Brier_SD_across_imputations": float(np.std([row["mean_Brier"] for row in per_imputation])), **params})
    table = pd.DataFrame(rows)
    table["Distance_to_Ideal"] = distance_to_ideal(table["C"], table["mean_Brier"])
    table = select_final(table)
    path = artifact_path("xgb_death_rescored", model_signature, ".pkl")
    save_pickle({"signature": model_signature, "table": table, "detail": detail}, path)
    return table
# Exercise both historical tie-breaks with a deliberately tied synthetic table.
tie_control = pd.DataFrame({"trial_id": [0, 1, 2], "Distance_to_Ideal": [0.2] * 3, "global_C": [0.70, 0.71, 0.71], "C_SD_across_imputations": [0.01, 0.02, 0.01]})
assert select_final(tie_control)["trial_id"].tolist() == [2, 1, 0]
print("Mortality search and both final tie-breaks are ready. No study is started in preflight.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Mortality search and both final tie-breaks are ready. No study is started in preflight.
Elapsed: 0.000 minutes


## Authorized execution stage
`preflight` performs checks without fitting cohort imputers or XGBoost models. `pilot` times one actual mortality fold with the historical winning parameters; its result is not reportable. `full` performs the complete accepted search and final reevaluation. Full-run model selection is never inferred from the timing pilot.

In [11]:
#| label: death-run-authorized-stage
#| message: false
t0 = time.perf_counter()
# Run only the explicitly configured and resource-reviewed computation stage.
pilot_result, final_table = None, None
if mode == "preflight":
    # Check every actual fold/imputation encoding before authorizing expensive model fits.
    encoding_audit = []
    for imputation in range(1, 6):
        for scheme in splits:
            for fold in range(1, 6):
                x_train, x_valid, y_train, y_valid, metadata = encoded_fold(scheme, fold, imputation)
                assert len(x_train) + len(x_valid) == len(development_data)
                assert not set(metadata_names + outcome_names).intersection(x_train.columns)
                encoding_audit.append({"Imputation": imputation, "Scheme": scheme, "Fold": fold, "Encoded predictors": x_train.shape[1], "Missing training": int(x_train.isna().sum().sum()), "Missing validation": int(x_valid.isna().sum().sum())})
    show_table(pd.DataFrame(encoding_audit), "All completed-data CV encodings verified without model fitting")
    del x_train, x_valid, y_train, y_valid, metadata
    print("Preflight passed. No imputer, XGBoost model or Optuna trial was fitted.")
elif mode == "pilot":
    pilot_result = evaluate_fold(native_params(historical_winner), "dual", 1, 1)
    show_table(pd.DataFrame({"Item": ["Purpose", "New imputation fits", "Model fitting time (minutes)", "Model plus scoring time (minutes)", "Returned boosting rounds", "Best iteration", "Encoded predictors"], "Value": ["Timing only; not reportable", 0, pilot_result["fit_seconds"] / 60, pilot_result["model_and_scoring_seconds"] / 60, pilot_result["returned_rounds"], pilot_result["best_iteration"], len(pilot_result["features"])]}), "Single-fold mortality timing pilot")
else:
    mortality_study = optimize_stage()
    show_table(pareto_table(mortality_study), "Mortality Pareto candidates, one cyclic imputation per trial")
    final_table = rescore_pareto(mortality_study)
    show_table(final_table, "Final mortality candidates: all five folds and all five imputations")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

trial_id,C,mean_Brier,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,Distance_to_Ideal
9,0.773732,0.017676,0.018255,4,20,0.825041,0.559776,0.003126,6.021455,0.432032,0.226957
16,0.773280,0.017675,0.015938,5,13,0.687919,0.627864,0.047831,13.910425,0.890525,0.227407
12,0.772976,0.017666,0.007869,4,24,0.802660,0.537321,0.052331,7.239097,0.337308,0.227711
95,0.772796,0.017664,0.032047,4,6,0.622623,0.463661,0.227006,5.402668,0.332106,0.227889
77,0.772404,0.017664,0.008135,4,21,0.840526,0.526540,0.245989,2.449769,0.166958,0.228281


trial_id,C,mean_Brier,global_C,C_SD_across_imputations,Brier_SD_across_imputations,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,gamma,Distance_to_Ideal
16,0.772858,0.017675,0.747943,0.000264,0.000003,0.015938,5,13,0.687919,0.627864,0.047831,13.910425,0.890525,0.227829
9,0.772805,0.017675,0.747433,0.000509,0.000003,0.018255,4,20,0.825041,0.559776,0.003126,6.021455,0.432032,0.227882
95,0.772582,0.017669,0.747551,0.000249,0.000003,0.032047,4,6,0.622623,0.463661,0.227006,5.402668,0.332106,0.228104
12,0.772378,0.017674,0.747675,0.000545,0.000004,0.007869,4,24,0.802660,0.537321,0.052331,7.239097,0.337308,0.228307
77,0.772021,0.017670,0.747712,0.000333,0.000004,0.008135,4,21,0.840526,0.526540,0.245989,2.449769,0.166958,0.228663


Elapsed: 0.002 minutes


## Historical versus clean-pipeline selection
Selected parameters changing at the precision of eTable 2 require a checkpoint. Tuning-score comparisons describe development model selection; they do not substitute for the later held-out or temporal evaluations. Changes cannot be attributed to imputation alone, because the zero-time label correction and deterministic sequential execution are also applied.

In [12]:
#| label: death-export-winner-and-comparison
#| message: false
t0 = time.perf_counter()
# Export only the final manuscript-facing hyperparameters and a reusable model-selection bundle.
parameter_comparison = pd.DataFrame()
if final_table is not None:
    winner = final_table.iloc[0]
    winner_params = native_params(winner)
    bundle_path = artifact_path("xgb_death_winner", model_signature, ".pkl")
    parameter_comparison = pd.DataFrame([{"Parameter": name, "Historical": native_params(historical_winner)[name], "Development_pipeline": winner_params[name], "Difference": winner_params[name] - native_params(historical_winner)[name], "Reported_decimals": 0 if name in ("max_depth", "min_child_weight") else 4, "Classification": "substantial" if round(winner_params[name], 0 if name in ("max_depth", "min_child_weight") else 4) != round(native_params(historical_winner)[name], 0 if name in ("max_depth", "min_child_weight") else 4) else "unchanged_at_reported_precision"} for name in param_names])
    summary = {"trial_id": int(winner["trial_id"]), **{name: float(winner[name]) for name in ["C", "mean_Brier", "global_C", "Distance_to_Ideal"]}}
    save_pickle({"signature": model_signature, "contract": model_contract, "params": winner_params, "summary": summary, "comparison": parameter_comparison, "final_candidates": final_table, "manuscript_checkpoint_required": True}, bundle_path)
    artifact_date = bundle_path.name.removeprefix("xgb_death_winner_").split("_")[0]
    table_path = table_dir / f"etable2_xgboost_mortality_{artifact_date}_{model_signature}.csv"
    manuscript_table = pd.DataFrame([{"Outcome": "Mortality", "Parameter": name, "Initial_search_lower": search_ranges[name][1], "Initial_search_upper": search_ranges[name][2], "Search_scale": "log" if search_ranges[name][3] else "linear", "Selected_value": winner_params[name], "Trial": int(winner["trial_id"]), "Source_cell": "death-export-winner-and-comparison"} for name in param_names])
    if table_path.exists():
        pd.testing.assert_frame_equal(pd.read_csv(table_path), manuscript_table, check_dtype=False)
    else:
        manuscript_table.to_csv(table_path, index=False)
        written.append(table_path)
    show_table(parameter_comparison, "eTable 2: historical versus development-imputed hyperparameters")
    metric_comparison = pd.DataFrame({"Metric": ["Mean horizon Uno C", "Mean horizon Brier", "Global Uno C"], "Historical": [float(historical_winner[c_column]), float(historical_winner[b_column]), float(historical_winner["Phase2_Global_C_Index"])], "Development_pipeline": [summary["C"], summary["mean_Brier"], summary["global_C"]]})
    metric_comparison["Difference"] = metric_comparison["Development_pipeline"] - metric_comparison["Historical"]
    show_table(metric_comparison, "Saved historical tuning references versus new final selection")
    print("Checkpoint required before notebook 03. Mortality retains its historical Breslow probability definition and mean horizon Brier.")
else:
    print("Final eTable 2 entries and winner bundle are pending full tuning; no pilot metric is exported as a manuscript result.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Parameter,Historical,Development_pipeline,Difference,Reported_decimals,Classification
learning_rate,0.030967,0.015938,-0.015029,4,substantial
max_depth,6.000000,5.000000,-1.000000,0,substantial
min_child_weight,15.000000,13.000000,-2.000000,0,substantial
subsample,0.778743,0.687919,-0.090824,4,substantial
colsample_bytree,0.616890,0.627864,0.010974,4,substantial
reg_alpha,0.013708,0.047831,0.034123,4,substantial
reg_lambda,2.536795,13.910425,11.373630,4,substantial
gamma,0.982286,0.890525,-0.091761,4,substantial


Metric,Historical,Development_pipeline,Difference
Mean horizon Uno C,0.762197,0.772858,0.010661
Mean horizon Brier,0.017730,0.017675,-0.000055
Global Uno C,0.745860,0.747943,0.002083


Checkpoint required before notebook 03. Mortality retains its historical Breslow probability definition and mean horizon Brier.
Elapsed: 0.002 minutes


## Interpreting the hyperparameter changes
Directions are shown separately because regularization, sampling and learning-rate changes may act in different directions. A parameter percentage change is not a percentage change in model complexity, convergence speed or predictive performance. The stopping summary describes the new winner only; historical learning curves have not been compared.
## Complete mortality run: comparison and checkpoint
The full run finished with 100 completed trials, five final Pareto candidates, 595 distinct model evaluations, zero failed trials and zero new imputation fits. Execution took 44.320 minutes; the sampled kernel-process-tree peak was 0.889 GiB with two native threads. The final winner's metrics were independently reaggregated from its 25 saved fold/imputation evaluations and its ranking was recomputed without fitting models.
| Tuning metric | Original saved selection | Current selection | Difference |
|---|---:|---:|---:|
| C | 0.762197 | 0.772858 | +0.010661 |
| mean_Brier | 0.017730 | 0.017675 | -0.000055 |
| global_C | 0.745860 | 0.747943 | +0.002083 |
All eight selected hyperparameters differ at the precision reported in eTable 2. The complete parameter comparison and direction table are displayed in the adjacent executed cells. The learning rate is lower, maximum depth is lower, and L1/L2 penalties are larger; the child-weight and gamma changes act in the opposite direction for allowing splits. These changes do not establish overall convergence speed or explain the metric differences by themselves.
### Verified comparison limits
The original and current C calculations use the complete validation fold at horizons 3, 6, 12, 36 and 60 months, then average horizons, folds and imputations. Reconstruction from the original Parquet, exclusions and saved split confirmed the same 70,521 development participants, endpoint values, programme values and row order; recreating the original fold helper yielded zero membership differences. The historical selection control reproduced trial 55 from its saved export; it did not refit the historical models.
The comparison is not an imputation-only experiment. In addition to the development-only imputations and retuning, the signed-time correction preserves 160 recorded deaths at time zero that the previous encoding did not present as events to XGBoost. Analytical times and participant inclusion remain unchanged. XGBoost identifies Cox events using a strictly positive signed label; see the [installed-version objective source](https://raw.githubusercontent.com/dmlc/xgboost/v3.1.3/src/objective/regression_obj.cu).
A further implementation difference is now explicit: the current encoder orders nominal levels deterministically and drops training-constant columns. Reconstructing the available original encoder produced the same informative columns and reference categories, plus the constant column tr_outcome_other. For example, the original cohabitation dummy order was family_of_origin, with_couple_children, others; the current order is others, family_of_origin, with_couple_children. Column order and removal of a constant column may affect seeded column subsampling; their contribution to the result has not been measured.
The original notebook also consumes imputations_tune without defining it in a preceding cell. Its exact historical in-memory feature schema cannot be established from that notebook alone. The current pipeline records its inputs and code explicitly; this does not justify claiming byte-for-byte reproduction of the historical training matrices.
These scores describe internal model selection. Imputation is shared across CV folds, so this is not independent validation of the complete preprocessing procedure. Holdout rows did not fit the imputers or select this winner. Changes to later SHAP rules, Cox specifications and Tables 2-4 remain unmeasured.
### Investigator checkpoint
Closed by the investigator after the fixed-parameter comparison: retain the selected mortality winner and current documented representation, and prepare notebook 03. The original executed comparison remains below as evidence for that decision.


In [13]:
#| label: death-interpret-hyperparameter-changes
#| message: false
t0 = time.perf_counter()
# Interpret parameter directions without changing the selected model or fitting new models.
if final_table is None:
    print("Interpretation is pending full tuning or reuse of its complete saved results.")
else:
    direction_notes = {
        "learning_rate": ("Larger contribution from each new tree, holding other settings fixed.", "Smaller contribution from each new tree, holding other settings fixed."),
        "max_depth": ("A greater maximum tree depth is allowed.", "A smaller maximum tree depth is allowed."),
        "min_child_weight": ("More restrictive threshold for splitting a node.", "Less restrictive threshold for splitting a node."),
        "subsample": ("More training rows are sampled for each tree.", "Fewer training rows are sampled for each tree."),
        "colsample_bytree": ("More predictor columns are available to each tree.", "Fewer predictor columns are available to each tree."),
        "reg_alpha": ("Stronger L1 penalty on leaf weights.", "Weaker L1 penalty on leaf weights."),
        "reg_lambda": ("Stronger L2 penalty on leaf weights.", "Weaker L2 penalty on leaf weights."),
        "gamma": ("A larger loss reduction is required before splitting.", "A smaller loss reduction is sufficient before splitting.")}
    interpretation = parameter_comparison[["Parameter", "Historical", "Development_pipeline", "Difference", "Classification"]].copy()
    interpretation["Relative_change_percent"] = 100 * interpretation["Difference"] / interpretation["Historical"].replace(0, np.nan)
    interpretation["Interpretation"] = ["Unchanged." if row.Difference == 0 else direction_notes[row.Parameter][0 if row.Difference > 0 else 1] for row in interpretation.itertuples()]
    interpretation = interpretation.rename(columns={"Historical": "Previous", "Development_pipeline": "Updated", "Relative_change_percent": "Relative change (%)"})
    show_table(interpretation[["Parameter", "Previous", "Updated", "Relative change (%)", "Interpretation", "Classification"]], "Direction and magnitude of the selected hyperparameter changes")
    # Describe the observed stopping behavior of the winning configuration using cached final evaluations.
    selected_runs = [row for row in score_cache["scores"].values() if row["scheme"] == "dual" and row["params"] == winner_params]
    assert len(selected_runs) == model_contract["budgets"]["folds"] * model_contract["budgets"]["imputations"]
    rounds = np.asarray([row["returned_rounds"] for row in selected_runs])
    show_table(pd.DataFrame([{"Final fold/imputation evaluations": len(selected_runs), "Minimum returned rounds": int(rounds.min()), "Median returned rounds": float(np.median(rounds)), "Maximum returned rounds": int(rounds.max()), "Configured round limit": model_contract["rounds"], "Evaluations reaching the limit": int((rounds == model_contract["rounds"]).sum())}]), "Observed stopping behavior of the selected configuration")
    print("Relative parameter changes are not percentage changes in training speed, model complexity or predictive performance.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Parameter,Previous,Updated,Relative change (%),Interpretation,Classification
learning_rate,0.030967,0.015938,-48.532190,"Smaller contribution from each new tree, holding other settings fixed.",substantial
max_depth,6.000000,5.000000,-16.666667,A smaller maximum tree depth is allowed.,substantial
min_child_weight,15.000000,13.000000,-13.333333,Less restrictive threshold for splitting a node.,substantial
subsample,0.778743,0.687919,-11.662879,Fewer training rows are sampled for each tree.,substantial
colsample_bytree,0.616890,0.627864,1.778990,More predictor columns are available to each tree.,substantial
reg_alpha,0.013708,0.047831,248.924087,Stronger L1 penalty on leaf weights.,substantial
reg_lambda,2.536795,13.910425,448.346353,Stronger L2 penalty on leaf weights.,substantial
gamma,0.982286,0.890525,-9.341553,A smaller loss reduction is sufficient before splitting.,substantial


Final fold/imputation evaluations,Minimum returned rounds,Median returned rounds,Maximum returned rounds,Configured round limit,Evaluations reaching the limit
25,350,420.000000,590,1500,0


Relative parameter changes are not percentage changes in training speed, model complexity or predictive performance.
Elapsed: 0.000 minutes


## Fixed historical parameters on current development inputs
The investigator authorized this additional diagnostic after inspecting the complete mortality tuning. The next cell reuses the existing evaluation helper, first reproduces the selected configuration from its 25 cached evaluations, then evaluates the historical parameter set with the same five completed development datasets, folds, corrected signed labels, encoder, probabilities and metric definitions. Tables remain in the notebook; fold scores reuse the existing dated cache. The selected winner and eTable 2 are not changed by this diagnostic.

In [14]:
#| label: death-compare-historical-parameters
#| message: false
t0 = time.perf_counter()
# Compare fixed parameter sets on identical current inputs; do not reopen the search or replace its winner.
fixed_parameter_diagnostic = None
if mode != "full" or final_table is None:
    print("The fixed-parameter comparison requires complete full-tuning artifacts.")
else:
    diagnostic_fits_before = new_model_fits
    diagnostic_metrics = ["C", "mean_Brier", "global_C"]
    diagnostic_runs = {}
    diagnostic_summaries = {}
    diagnostic_params = {"Selected parameters / current inputs": winner_params, "Historical parameters / current inputs": native_params(historical_winner)}
    for configuration, params in diagnostic_params.items():
        records = []
        for imputation in range(1, 6):
            records.extend(evaluate_fold(params, "dual", fold, imputation) for fold in range(1, 6))
            print(f"{configuration}: imputation {imputation}/5 complete", flush=True)
        assert len(records) == 25 and len({(row["imputation"], row["fold"]) for row in records}) == 25
        assert all(not row["imputation_refitted"] and row["imputation_fit_scope"] == "complete_development_before_cv" for row in records)
        diagnostic_runs[configuration] = records
        per_imputation = pd.DataFrame(records).groupby("imputation")[diagnostic_metrics].mean()
        diagnostic_summaries[configuration] = {name: float(per_imputation[name].mean()) for name in diagnostic_metrics}
        diagnostic_summaries[configuration]["C_SD_across_imputations"] = float(np.std(per_imputation["C"].to_numpy()))
        diagnostic_summaries[configuration]["Distance_to_Ideal"] = float(distance_to_ideal(diagnostic_summaries[configuration]["C"], diagnostic_summaries[configuration]["mean_Brier"]))
        if configuration == "Selected parameters / current inputs":
            # Reproduce the saved selected result before trusting any newly fitted comparison.
            assert new_model_fits == diagnostic_fits_before, "The selected configuration must reuse all 25 saved evaluations."
            for name in diagnostic_metrics + ["Distance_to_Ideal", "C_SD_across_imputations"]:
                assert np.isclose(diagnostic_summaries[configuration][name], float(winner[name]), atol=1e-14, rtol=0), name
    selected_summary = diagnostic_summaries["Selected parameters / current inputs"]
    historical_summary = diagnostic_summaries["Historical parameters / current inputs"]
    original_summary = {"C": float(historical_winner[c_column]), "mean_Brier": float(historical_winner[b_column]), "global_C": float(historical_winner["Phase2_Global_C_Index"])}
    diagnostic_table = pd.DataFrame([{"Metric": name, "Historical saved export": original_summary[name], "Historical parameters / current inputs": historical_summary[name], "Selected parameters / current inputs": selected_summary[name], "Difference between parameters on current inputs": selected_summary[name] - historical_summary[name]} for name in diagnostic_metrics])
    show_table(diagnostic_table, "Fixed mortality parameters: historical export and matched current-input comparison")
    # Pair every fold and imputation, including the exact encoded feature order.
    selected_records = diagnostic_runs["Selected parameters / current inputs"]
    historical_records = diagnostic_runs["Historical parameters / current inputs"]
    assert all((a["imputation"], a["fold"], a["features"], a["imputation_source"]) == (b["imputation"], b["fold"], b["features"], b["imputation_source"]) for a, b in zip(selected_records, historical_records))
    horizon_rows = []
    for metric, key in [("Uno C", "C_by_horizon"), ("Brier", "Brier_by_horizon")]:
        current_values = np.asarray([row[key] for row in selected_records]).mean(axis=0)
        historical_values = np.asarray([row[key] for row in historical_records]).mean(axis=0)
        for horizon, previous, current in zip(horizons, historical_values, current_values):
            horizon_rows.append({"Metric": metric, "Horizon (months)": float(horizon), "Historical parameters / current inputs": float(previous), "Selected parameters / current inputs": float(current), "Difference": float(current - previous)})
    show_table(pd.DataFrame(horizon_rows), "Per-horizon comparison on the same five imputations and folds")
    diagnostic_ranking = select_final(pd.DataFrame([{"Configuration": name, **values} for name, values in diagnostic_summaries.items()]))
    show_table(diagnostic_ranking, "Diagnostic ranking under the unchanged distance and tie-breaks; saved winner remains unchanged")
    fixed_parameter_diagnostic = {"model_signature": model_signature, "data_signature": data_signature, "historical_trial_id": int(historical_winner["trial_id"]), "selected_trial_id": int(winner["trial_id"]), "parameters": diagnostic_params, "summaries": diagnostic_summaries, "historical_saved_export": original_summary, "horizons": horizon_rows, "preferred_fixed_configuration": diagnostic_ranking.iloc[0]["Configuration"], "selected_control_evaluations": 25, "selected_control_tolerance": 1e-14, "new_model_fits": new_model_fits - diagnostic_fits_before, "new_imputation_fits": 0, "selected_artifact_replaced": False, "elapsed_seconds": time.perf_counter() - t0}
    print("FIXED_PARAMETER_DIAGNOSTIC " + json.dumps(fixed_parameter_diagnostic, sort_keys=True))
    print("This comparison isolates the difference between these parameter sets on current inputs, not the separate effects of imputation, zero-time labels or encoding.")
    print("Internal CV shares development-wide imputation. No holdout result, independent preprocessing validation, confidence interval or model superiority claim is produced.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Selected parameters / current inputs: imputation 1/5 complete


Selected parameters / current inputs: imputation 2/5 complete


Selected parameters / current inputs: imputation 3/5 complete


Selected parameters / current inputs: imputation 4/5 complete


Selected parameters / current inputs: imputation 5/5 complete


...

Historical parameters / current inputs: imputation 5/5 complete


Metric,Historical saved export,Historical parameters / current inputs,Selected parameters / current inputs,Difference between parameters on current inputs
C,0.762197,0.771221,0.772858,0.001637
mean_Brier,0.017730,0.017688,0.017675,-0.000012
global_C,0.745860,0.747128,0.747943,0.000815


Metric,Horizon (months),Historical parameters / current inputs,Selected parameters / current inputs,Difference
Uno C,3.000000,0.789165,0.791149,0.001985
Uno C,6.000000,0.770619,0.772887,0.002268
Uno C,12.000000,0.761474,0.762873,0.001399
Uno C,36.000000,0.767924,0.769534,0.001610
Uno C,60.000000,0.766925,0.767846,0.000921
Brier,3.000000,0.004854,0.004858,0.000004
Brier,6.000000,0.006825,0.006831,0.000006
Brier,12.000000,0.010447,0.010450,0.000003
Brier,36.000000,0.026454,0.026413,-0.000041
Brier,60.000000,0.039858,0.039824,-0.000034


Configuration,C,mean_Brier,global_C,C_SD_across_imputations,Distance_to_Ideal
Selected parameters / current inputs,0.772858,0.017675,0.747943,0.000264,0.227829
Historical parameters / current inputs,0.771221,0.017688,0.747128,0.000841,0.229461


FIXED_PARAMETER_DIAGNOSTIC {"data_signature": "2d41a3d8065b", "elapsed_seconds": 53.42993410001509, "historical_saved_export": {"C": 0.7621969298890017, "global_C": 0.7458596163068426, "mean_Brier": 0.0177297590973591}, "historical_trial_id": 55, "horizons": [{"Difference": 0.001984844828714083, "Historical parameters / current inputs": 0.7891646458044019, "Horizon (months)": 3.0, "Metric": "Uno C", "Selected parameters / current inputs": 0.791149490633116}, {"Difference": 0.002267774442959891, "Historical parameters / current inputs": 0.7706192821868801, "Horizon (months)": 6.0, "Metric": "Uno C", "Selected parameters / current inputs": 0.77288705662984}, {"Difference": 0.0013994413932305605, "Historical parameters / current inputs": 0.7614735945237746, "Horizon (months)": 12.0, "Metric": "Uno C", "Selected parameters / current inputs": 0.7628730359170052}, {"Difference": 0.0016099657736940998, "Historical parameters / current inputs": 0.767923864955458, "Horizon (months)": 36.0, "Met

## Fixed-parameter mortality comparison completed (2026-10-03)
On the same current development inputs, the historical parameter set gives mean horizon C = 0.771221, compared with 0.772858 for the selected set (difference +0.001637). C is higher for the selected set at all five horizons. Mean horizon Brier changes from 0.017688 to 0.017675 (difference -0.000012); this is an arithmetic mean, not a time-integrated Brier score. The Brier change is not uniform: it increases slightly at 3, 6 and 12 months and decreases at 36 and 60 months. Global Uno C changes from 0.747128 to 0.747943.
The unchanged selection criterion still ranks the selected configuration first: distance to the ideal point is 0.227829, versus 0.229461 for the historical parameters on current inputs. This diagnostic does not reverse the choice. The winner bundle, saved Optuna study and eTable 2 export retain identical SHA-256 hashes. The investigator has accepted the eight previously documented eTable 2 parameter changes and the current representation; manuscript integration remains separate.
The historical saved C is 0.762197. Keeping the historical parameters while using current inputs yields a difference of +0.009024 from that reference; the difference between parameter sets on current inputs is +0.001637. Thus most of the observed original-to-current difference is already present before changing parameters. The first contrast combines changes in development imputation, signed zero-time labels, encoded representation and other historical execution conditions; their individual contributions remain unmeasured. It must not be presented as the isolated effect of imputation or removal of leakage.
Control: the selected configuration was reaggregated from all 25 saved evaluations before fitting the historical configuration, with absolute tolerance 1e-14. The diagnostic fitted 25 additional models, zero imputers and zero Optuna trials; all 17 code cells completed without errors. Full notebook execution took 1.043 minutes, including 0.890 minutes in the comparison cell; the sampled kernel-process-tree peak was 0.768 GiB with two native threads. The original 44.320-minute tuning record is preserved separately.
These are descriptive internal CV comparisons using development-wide imputation shared across folds. No confidence interval, holdout performance estimate, general superiority claim or clinical-benefit claim follows from these point differences. The adoption checkpoint is now closed; preparation of notebook 03 is authorized, while full SHAP computation awaits its resource review.

## Resources and execution status
Native thread pools are capped at two and Optuna runs one trial at a time. The imputation models are reused through their completed Parquet outputs; no R subprocess or imputation fit runs here. The runner records elapsed time and sampled process-tree peak RAM. A completed preflight is not a completed tuning run.

In [15]:
#| label: death-resource-and-status
#| message: false
t0 = time.perf_counter()
# Report current resources; the external runner records the sampled process-tree peak.
encoded_fold.cache_clear()
completed_development.cache_clear()
gc.collect()
show_table(pd.DataFrame({"Item": ["Mode", "Completed development inputs", "New cohort model fits in this execution", "Full tuning complete", "Current Python RSS (GiB)", "Available RAM (GiB)", "Native threads", "Elapsed after setup (minutes)"], "Value": [mode, "verified", new_model_fits, final_table is not None, round(process.memory_info().rss / 2**30, 3), round(psutil.virtual_memory().available / 2**30, 3), threads, round((time.perf_counter() - run_started) / 60, 3)]}), "Execution status")
if mode == "preflight":
    print("Preflight estimates no tuning performance. Full mode runs the authorized search after its resource review.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Item,Value
Mode,full
Completed development inputs,verified
New cohort model fits in this execution,25
Full tuning complete,True
Current Python RSS (GiB),0.706000
Available RAM (GiB),53.941000
Native threads,2
Elapsed after setup (minutes),0.972000


Elapsed: 0.001 minutes


## Session info

In [16]:
#| label: death-session-info
#| message: false
t0 = time.perf_counter()
# Record the actual Python environment and native thread limits.
package_names = ["numpy", "pandas", "xgboost", "optuna", "scikit-learn", "scikit-survival", "pyarrow", "PyYAML", "psutil", "threadpoolctl", "joblib"]
show_table(pd.DataFrame({"Package": package_names, "Version": [importlib.metadata.version(name) for name in package_names]}), "Installed execution packages")
show_table(pd.DataFrame([{"Library": row.get("internal_api"), "Threads": row.get("num_threads"), "Version": row.get("version")} for row in threadpool_info()]), "Native thread pools")
assert all(row["num_threads"] <= threads for row in threadpool_info())
print("Imputation provenance: notebook 00 used R/r-main. This tuning session uses Python/surv only.")
print(f"Python {platform.python_version()} | {platform.system()} {platform.release()} | seed {seed}")
print(f"Elapsed since setup: {(time.perf_counter() - run_started) / 60:.3f} minutes")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Package,Version
numpy,2.0.1
pandas,3.0.0
xgboost,3.1.3
optuna,4.7.0
scikit-learn,1.8.0
scikit-survival,0.27.0
pyarrow,15.0.2
PyYAML,6.0.3
psutil,7.2.2
threadpoolctl,3.6.0


Library,Threads,Version
mkl,2,2023.1-Product
openmp,2,NaN
openmp,2,NaN


Imputation provenance: notebook 00 used R/r-main. This tuning session uses Python/surv only.
Python 3.11.14 | Windows 10 | seed 2125
Elapsed since setup: 0.975 minutes
Elapsed: 0.001 minutes


## Files written and checkpoint
Only final manuscript-facing tables and reusable computation objects are exported. Exploratory comparisons remain in this notebook and its embedded-resource HTML.

In [17]:
#| label: death-written-files-and-status
#| message: false
t0 = time.perf_counter()
# List artifacts actually written or updated by this execution, using project-relative paths.
manifest = pd.DataFrame([{"Path": path.relative_to(root).as_posix(), "Bytes": path.stat().st_size} for path in dict.fromkeys(written)], columns=["Path", "Bytes"])
show_table(manifest, "Files written in outputs")
execution_status = {"mode": mode, "completed_development_inputs_verified": True, "imputation_fit_scope": "complete_development_before_cv", "new_imputation_fits": 0, "full_tuning_complete": final_table is not None, "new_model_fits": new_model_fits, "data_signature": data_signature, "model_signature": model_signature, "checkpoint_before_notebook_03": True}
print(json.dumps(execution_status, sort_keys=True))
print("Full mortality tuning finished; review the comparison before notebook 03." if final_table is not None else "Mortality tuning remains pending; no new final hyperparameters are reported.")
print(f"Elapsed: {(time.perf_counter() - t0) / 60:.3f} minutes")

Path,Bytes
outputs/objects/xgb_death_rescored_2026-10-03_1b598df5b837.pkl,2977
outputs/objects/xgb_death_winner_2026-10-03_1b598df5b837.pkl,49849
outputs/objects/xgb_death_fold_scores_2026-10-03_1b598df5b837.pkl,1136207


{"checkpoint_before_notebook_03": true, "completed_development_inputs_verified": true, "data_signature": "2d41a3d8065b", "full_tuning_complete": true, "imputation_fit_scope": "complete_development_before_cv", "mode": "full", "model_signature": "1b598df5b837", "new_imputation_fits": 0, "new_model_fits": 25}
Full mortality tuning finished; review the comparison before notebook 03.
Elapsed: 0.000 minutes
